### Rotation – Lösung Schritt für Schritt

*35. Bundeswettbewerb Informatik (2016), 1. Runde, Aufgabe 3*

Ein mechanisches Puzzle besteht aus einem quadratischen Rahmen, der Stäbchen verschiedener Farben enthält und an einer Stelle offen ist. Der Rahmen kann jeweils um 90° nach links oder rechts gedreht werden. Durch die Schwerkraft gleiten dann die Stäbchen nach unten, ohne ihre Ausrichtung zu verändern, bis sie auf ein Hindernis stoßen. Ziel ist es, irgendein Stäbchen mit möglichst wenig Drehungen aus dem Rahmen herauszubekommen.

Im Beispiel der Aufgabe sieht man, wie die Stäbchen einer Puzzleaufgabe auf sechs aufeinanderfolgende Drehungen reagieren. Am Ende gleitet ein Stäbchen aus dem Rahmen.

Es kann aber auch vorkommen, dass es keine Möglichkeit gibt, irgendein Stäbchen herauszubekommen, da immer etwas im Weg ist.

#### Aufgabe 3

Eine Puzzleaufgabe ist durch die Anzahl und anfängliche Lage der Stäbchen im Rahmen, die Größe des Rahmens und die Lage der offenen Stelle im Rahmen gegeben. Schreibe ein Programm, das eine solche Aufgabe aus einer Datei einlesen kann und dann entweder löst und dazu eine kürzestmögliche Folge von Drehungen ausgibt oder meldet, dass es keine Lösungsmöglichkeit gibt.

Wende dein Programm mindestens auf die drei Beispiele `rotation1.txt` bis `rotation3.txt` an und dokumentiere jeweils das Ergebnis.

**Zu den Beispieldaten:** Es gibt keine README. Der Aufbau ist aus den Dateien abgelesen: Die erste Zeile enthält die Größe `n` des Rahmens (mit Rand), dann folgen `n` Zeilen. `#` ist der Rahmen, ein Leerzeichen ein freies Feld, eine Lücke im Rand die offene Stelle, und eine Ziffer ein Teil eines Stäbchens (gleiche Ziffer = dasselbe Stäbchen). Beispiel `rotation1.txt`:

```
8
########
#     0#
#     0#
#112222#
#33   4#
#55   4#
#666  4#
### ####
```

Im Ordner `eigene_testdaten` liegen zwei kleine Puzzle: `eine_rotation.txt` (eine Rotation genügt) und `ohne_loesung.txt` (nach dem Puzzle ohne Lösung aus den Lösungshinweisen; die Leerzeichen habe ich dort aus dem Bild abgeschätzt).

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Zustände und Übergänge:** Das Puzzle hat in jedem Moment einen *Zustand* (wo liegt welches Stäbchen). Aus jedem Zustand führen genau zwei Übergänge heraus: „nach links drehen“ und „nach rechts drehen“. Eine Aufgabe dieser Art ist ein **Weg in einem Graphen**, dessen Ecken die Zustände sind.
- **Breitensuche (BFS):** Sie probiert erst alle Zustände nach 0 Drehungen, dann alle nach 1 Drehung, dann nach 2 usw. Der erste Treffer hat deshalb automatisch die **kleinste Zahl von Drehungen**. Eine Tiefensuche (immer weiter in eine Richtung) findet dagegen irgendeine, meist viel zu lange Lösung.
- **Schon besuchte Zustände merken:** Dreht man links und gleich wieder rechts, ist man wieder im alten Zustand. Ohne „Merkliste“ würde die Suche ewig dasselbe wiederholen und bei einem Puzzle ohne Lösung nie enden. Mit Merkliste endet sie, sobald alle erreichbaren Zustände gesehen sind.
- **Den Weg rekonstruieren:** Zu jedem Zustand merkt man sich den *Vorgänger* und die Drehung, mit der man ihn erreicht hat. Vom Ziel aus geht man rückwärts bis zum Start.
- **Simulation der Schwerkraft:** Stäbchen, die schon unten liegen, müssen zuerst fertig sein, bevor die darüber nachrutschen. Man bearbeitet die Stäbchen daher **von unten nach oben**.

**Nützliche Funktionen und Bibliotheken**
- `collections.deque`: eine Warteschlange, bei der man vorne schnell etwas entnimmt (`popleft`) und hinten anfügt (`append`). Genau das braucht die Breitensuche.
- **Dictionary** (`dict`): ordnet jedem Zustand seinen Vorgänger zu und merkt gleichzeitig, welche Zustände schon gesehen sind (`in` ist schnell).
- **Tupel von Zeichenketten** als Zustand: Listen kann man nicht als Schlüssel eines Dictionary verwenden, Tupel schon. Der Rahmen steht also als Tupel von Zeilen, jede Zeile ein String.
- `zip(*zeilen)`: macht aus den Zeilen einer Tabelle ihre Spalten (die Tabelle wird „gekippt“). Mit `[::-1]` (Reihenfolge umkehren) ergibt sich daraus eine Drehung um 90°.

## Die Idee in Worten

1. Ein **Zustand** ist das Bild des Rahmens: `n` Zeilen mit `#`, Leerzeichen und Ziffern. Den Rahmen drehen wir immer **mit seinem Inhalt** (so sieht man es im Bild der Aufgabe): Nach der Drehung steht der Rahmen wieder aufrecht, die Lücke ist an einer anderen Stelle, und die Stäbchen liegen anders. Dass die Stäbchen „ihre Ausrichtung behalten“, bedeutet in dieser Sicht: Ein senkrechtes Stäbchen liegt nach einer Drehung waagerecht, denn der Rahmen hat sich gedreht, nicht das Stäbchen.
2. **Eine Rotation** = Bild um 90° drehen und dann alle Stäbchen **fallen lassen**, bis nichts mehr nach unten rutschen kann.
3. **Ziel:** Ein Stäbchen steht in der Lücke des Rahmens. Das geht nur, wenn die Lücke unten ist und das Stäbchen senkrecht darüber steht (ein waagerechtes Stäbchen der Länge 2 passt nicht durch eine Lücke der Breite 1). Es fällt dann heraus. Das ist von selbst so, wenn wir die Lücke wie ein **freies Feld im Rand** behandeln.
4. **Breitensuche** vom Startzustand aus. Jeder Zustand, den wir zum ersten Mal sehen, kommt in die Warteschlange. Erreichen wir einen Zielzustand, geben wir den Weg aus. Wird die Warteschlange leer, ohne dass wir ein Ziel gefunden haben, gibt es keine Lösung.

**Ein Gedanke zur Größe:** Nach 90 Drehungen gibt es $2^{90}$ mögliche Folgen, das ist mehr als $10^{27}$. Die Breitensuche schafft das trotzdem, weil sich die Zustände ständig wiederholen: Es gibt nur wenige verschiedene. Am Ende sehen wir, wie viele.

## Schritt 1: Datei einlesen

Wir lesen die Datei in ein Tupel von Zeichenketten. Manche Editoren schneiden Leerzeichen am Zeilenende ab (das bräuchten die Lücken im rechten Rand), deshalb füllen wir jede Zeile mit `ljust(n)` auf die Breite `n` auf. `zeige` druckt ein Puzzle.

In [ ]:
def lies_puzzle(dateiname):
    """Liest ein Puzzle: Rückgabe ist ein Tupel von n Zeichenketten der Länge n."""
    with open(dateiname) as datei:
        zeilen = datei.read().split('\n')
    n = int(zeilen[0])
    return tuple(zeilen[1 + i].ljust(n) for i in range(n))


def zeige(puzzle):
    """Druckt ein Puzzle; die Ränder sehen so aus wie in der Datei."""
    for zeile in puzzle:
        print(zeile)


start1 = lies_puzzle('beispieldaten/rotation1.txt')
zeige(start1)

########
#     0#
#     0#
#112222#
#33   4#
#55   4#
#666  4#
### ####


## Schritt 2: Den Rahmen drehen

`zip(*puzzle)` macht aus den Zeilen die **Spalten** (die erste Spalte von oben nach unten ist das erste Element). Daraus entstehen die beiden Drehungen:

- **Rotation nach links** (gegen den Uhrzeigersinn): Die **rechte** Spalte wird zur obersten Zeile, die oberste Zeile zur linken Spalte. Wir nehmen die Spalten von `zip(*puzzle)` und kehren ihre Reihenfolge um (`[::-1]`): Die letzte Spalte wird die erste Zeile.
- **Rotation nach rechts** (im Uhrzeigersinn): Die **linke** Spalte wird zur obersten Zeile, die unterste Zeile zur linken Spalte. Wir kehren zuerst die Zeilen um (`puzzle[::-1]`) und bilden dann die Spalten.

Die `zip`-Spalten sind Tupel einzelner Zeichen, deshalb setzen wir sie mit `''.join` wieder zu Strings zusammen. In der Ausgabe ist noch nichts gefallen: Die Rotation allein ist nur das gedrehte Bild.

In [ ]:
def dreh_links(puzzle):
    """Bild um 90 Grad gegen den Uhrzeigersinn drehen (noch ohne Schwerkraft)."""
    spalten = list(zip(*puzzle))
    return tuple(''.join(spalte) for spalte in spalten[::-1])


def dreh_rechts(puzzle):
    """Bild um 90 Grad im Uhrzeigersinn drehen (noch ohne Schwerkraft)."""
    spalten = zip(*puzzle[::-1])
    return tuple(''.join(spalte) for spalte in spalten)


zeige(dreh_links(start1))
print()
zeige(dreh_rechts(start1))

########
#002444#
#  2   #
#  2   #
#  2  6 
#  1356#
#  1356#
########

########
#6531  #
#6531  #
 6  2  #
#   2  #
#   2  #
#444200#
########


In der ersten Ausgabe (links gedreht) ist die Lücke von unten nach rechts gewandert und das Stäbchen `0` von der rechten Seite nach oben, jetzt waagerecht. In der zweiten (rechts gedreht) wanderte die Lücke nach links.

## Schritt 3: Die Stäbchen fallen lassen

Jede Ziffer steht für ein Stäbchen. Zuerst suchen wir zu jeder Ziffer **alle Felder**, die sie belegt (Dictionary `zellen`: Ziffer → Liste von `(zeile, spalte)`). Weil wir die Zeilen von oben nach unten durchgehen, steht in jeder Liste das **unterste Feld zuletzt**.

Dann lassen wir die Stäbchen **von unten nach oben** fallen: Wir sortieren die Ziffern nach der Zeile ihres untersten Feldes, das unterste Stäbchen kommt zuerst. Ein Stäbchen rutscht eine Zeile nach unten, solange **alle** seine Felder darunter frei sind. „Frei“ heißt: ein Leerzeichen, oder ein Feld, das zum Stäbchen selbst gehört (ein senkrechtes Stäbchen steht ja über seinem eigenen Feld). Rutscht ein Stäbchen, werden seine alten Felder frei und die neuen belegt.

Weil alle Stäbchen *unter* dem aktuellen schon fertig liegen, genügt ein Durchgang. Die Lücke im Rand ist ein Leerzeichen. Ein senkrechtes Stäbchen über der Lücke rutscht also noch in sie hinein. Die Zeile darunter gibt es nicht mehr (`r + 1 < n` ist falsch): Dort bleibt es „in der Lücke stehen“, und das bedeutet für uns: **herausgefallen**.

Die Sortierung ist ein Python-Detail, das über die Grundlagen hinausgeht: `sorted(ziffern, key=unterste_zeile)` sortiert die Ziffern nach dem Wert, den die kleine Hilfsfunktion `unterste_zeile` für jede Ziffer zurückgibt. Mit dem Minuszeichen davor kommt der größte Wert (das tiefste Stäbchen) zuerst. (Wir übergeben hier eine Funktion; für Neulinge reicht es zu wissen: „sortiere nach diesem Wert“.)

In [ ]:
def fallen(puzzle):
    """Lässt alle Stäbchen nach unten fallen und gibt das neue Puzzle zurück."""
    n = len(puzzle)
    zellen = {}                                   # Ziffer -> Felder des Stäbchens
    for r in range(n):
        for c in range(n):
            ziffer = puzzle[r][c]
            if ziffer not in ' #':
                if ziffer not in zellen:
                    zellen[ziffer] = []
                zellen[ziffer].append((r, c))

    gitter = [list(zeile) for zeile in puzzle]    # veränderbare Kopie

    def unterste_zeile(ziffer):
        return -zellen[ziffer][-1][0]             # Minus: unterstes Stäbchen zuerst

    for ziffer in sorted(zellen, key=unterste_zeile):
        felder = zellen[ziffer]
        # solange alle Felder darunter frei (oder eigene) sind: eine Zeile tiefer
        while all(r + 1 < n and gitter[r + 1][c] in ' ' + ziffer for r, c in felder):
            for r, c in felder:
                gitter[r][c] = ' '
            felder = [(r + 1, c) for r, c in felder]
            for r, c in felder:
                gitter[r][c] = ziffer
    return tuple(''.join(zeile) for zeile in gitter)


def drehung(puzzle, richtung):
    """Eine Rotation: 'L' oder 'R', danach fallen lassen."""
    if richtung == 'L':
        return fallen(dreh_links(puzzle))
    return fallen(dreh_rechts(puzzle))


nach1 = drehung(start1, 'L')
zeige(nach1)

########
#  2   #
#  2   #
#  2444#
#  2  6 
#  1356#
#001356#
########


Das ist das zweite Bild aus den Lösungshinweisen (erste Rotation nach links bei `rotation1.txt`): Das Stäbchen `0` ist nach unten links gerutscht und liegt jetzt waagerecht, die Stäbchen `1` bis `6` stehen aufeinander und nebeneinander. Die Lücke ist rechts, die `6` steht direkt daneben. Sie fällt nicht hinaus: Die Schwerkraft zieht nur nach unten, und die Lücke liegt seitlich.

Wir überprüfen die ersten beiden Schritte gegen die Lösungshinweise (Rotation nach links, dann noch einmal nach links):

In [ ]:
erwartet_1 = ('########',
              '#  2   #',
              '#  2   #',
              '#  2444#',
              '#  2  6 ',
              '# 1356 #',
              '#001356#',
              '########')
# Die Leerzeichen der Lösungshinweise sind im PDF-Text verloren gegangen, das Bild dort ist maßgeblich.
# Wir vergleichen daher nur die Ziffern und die Lücke Zeile für Zeile ohne Leerzeichen.
def ohne_leer(puzzle):
    return tuple(zeile.replace(' ', '') for zeile in puzzle)


print(ohne_leer(nach1) == ohne_leer(erwartet_1))
nach2 = drehung(nach1, 'L')
zeige(nach2)

True
#### ###
#   666#
#    55#
#  4 33#
#  4 11#
#  4  0#
#2222 0#
########


Auch hier stimmt die Reihenfolge der Ziffern mit dem dritten Bild der Lösungshinweise überein (die Lücke ist jetzt oben, deshalb kann nichts hinausfallen).

## Schritt 4: Ist ein Stäbchen herausgefallen?

Ein Stäbchen ist „draußen“, sobald eines seiner Felder im **Rand** steht, also in der obersten oder untersten Zeile oder in der ersten oder letzten Spalte. Dort kann außer der Lücke nur ein Zeichen `#` stehen. Eine Ziffer im Rand heißt deshalb: Ein Stäbchen steht in der Lücke.

In [ ]:
def ist_fertig(puzzle):
    """Wahr, wenn irgendein Stäbchen in der Lücke des Rahmens steht."""
    n = len(puzzle)
    for i in range(n):
        for ch in (puzzle[0][i], puzzle[n - 1][i], puzzle[i][0], puzzle[i][n - 1]):
            if ch not in ' #':
                return True
    return False


print(ist_fertig(start1), ist_fertig(nach1))

False False


## Schritt 5: Breitensuche

Wir halten zwei Dinge:

- `vorgaenger`: Dictionary `Zustand -> (Vorgänger, Drehung)`. Der Startzustand hat keinen Vorgänger (`None`). Wer in diesem Dictionary steht, ist **schon gesehen**.
- `warteschlange`: die Zustände, deren Folgezustände wir noch ausprobieren müssen. Wir nehmen immer den *ältesten* vorne heraus, und was neu ist, hängen wir hinten an. Dadurch sind erst alle Zustände nach `k` Drehungen dran, ehe die nach `k + 1` Drehungen kommen.

Ist der Zustand, den wir gerade vorne herausnehmen, ein Ziel (`ist_fertig`), dann gehen wir über die Vorgänger rückwärts und setzen die Drehungen zu einem String wie `LLLLRL` zusammen. Gibt es kein Ziel, endet die `while`-Schleife von selbst, weil die Warteschlange irgendwann leer ist.

Die Funktion gibt neben der Drehfolge die Zahl der **gesehenen Zustände** zurück (damit wir die Größe des Suchraums sehen).

In [ ]:
from collections import deque


def loese(start):
    """Kürzeste Drehfolge (String aus L und R) und Zahl der gesehenen Zustände.
    Gibt None statt der Drehfolge zurück, wenn es keine Lösung gibt."""
    vorgaenger = {start: None}
    warteschlange = deque([start])
    while len(warteschlange) > 0:
        zustand = warteschlange.popleft()
        if ist_fertig(zustand):
            folge = ''
            while vorgaenger[zustand] is not None:
                zustand, drehrichtung = vorgaenger[zustand]
                folge = drehrichtung + folge
            return folge, len(vorgaenger)
        for richtung in 'LR':
            neu = drehung(zustand, richtung)
            if neu not in vorgaenger:
                vorgaenger[neu] = (zustand, richtung)
                warteschlange.append(neu)
    return None, len(vorgaenger)


folge1, anzahl1 = loese(start1)
print('Beispiel 1:', folge1, '=', len(folge1), 'Rotationen,', anzahl1, 'Zustände gesehen')

Beispiel 1: LLLLRL = 6 Rotationen, 47 Zustände gesehen


Die Lösungshinweise geben für `rotation1.txt` **6 Rotationen** an: links, links, links, links, rechts, links. Das ist `LLLLRL`. Die Zahl der gesehenen Zustände ist winzig (47), obwohl es $2^6 = 64$ Folgen der Länge 6 gibt.

## Schritt 6: Die Lösung vorführen

Wir geben die Lösung so aus wie in den Lösungshinweisen: Startzustand, dann nach jeder Rotation das Bild. Bei langen Lösungen zeigen wir nur die Rotationsfolge.

In [ ]:
def zeige_loesung(start, ausfuehrlich):
    """Löst das Puzzle und druckt Rotationen (und auf Wunsch jedes Bild)."""
    folge, anzahl = loese(start)
    if folge is None:
        print('Es gibt keine Lösung! Es wurden alle', anzahl, 'Zustände ausprobiert.')
        return folge
    print('Es wurde eine Lösung mit', len(folge), 'Rotationen gefunden:', folge)
    if ausfuehrlich:
        zustand = start
        zeige(zustand)
        for richtung in folge:
            zustand = drehung(zustand, richtung)
            print('\nRotation nach', 'links:' if richtung == 'L' else 'rechts:')
            zeige(zustand)
    return folge


zeige_loesung(start1, True)

Es wurde eine Lösung mit 6 Rotationen gefunden: LLLLRL
########
#     0#
#     0#
#112222#
#33   4#
#55   4#
#666  4#
### ####

Rotation nach links:
########
#  2   #
#  2   #
#  2444#
#  2  6 
#  1356#
#001356#
########

Rotation nach links:
#### ###
#   666#
#    55#
#  4 33#
#  4 11#
#  4  0#
#2222 0#
########

Rotation nach links:
########
#      #
#    00#
      2#
#6 31 2#
#6531 2#
#654442#
########

Rotation nach links:
########
#  2222#
#     4#
#   114#
#   334#
# 0  55#
# 0 666#
### ####

Rotation nach rechts:
########
#      #
#      #
 00   2#
#6 31 2#
#6531 2#
#654442#
########

Rotation nach links:
########
#  2222#
#     4#
#   114#
#   334#
#    55#
#  0666#
###0####


Im letzten Bild steht die `0` in der Lücke unten: Sie fällt heraus. Das stimmt mit dem Beispiel der Lösungshinweise überein (dort sind die Bilder nach den Drehungen Zeichen für Zeichen gleich, die Leerzeichen gingen im PDF-Text verloren).

## Alle Beispiele

`rotation2.txt` ist ein 12×12-Rahmen mit neun Stäbchen. `rotation3.txt` ist der große Fall (10×10, zehn Stäbchen); die Suche dafür braucht etwas Geduld, weil sie viele Zustände sieht (im Browser dauert sie länger als auf dem eigenen Rechner). Die Zahl der Zustände steht jeweils dabei.

In [ ]:
ergebnisse = {}
for nummer in (1, 2, 3):
    start = lies_puzzle('beispieldaten/rotation' + str(nummer) + '.txt')
    folge, anzahl = loese(start)
    ergebnisse[nummer] = (folge, anzahl)
    print('rotation' + str(nummer) + '.txt:', len(folge), 'Rotationen,', anzahl, 'Zustände gesehen')
    print(folge)

rotation1.txt: 6 Rotationen, 47 Zustände gesehen
LLLLRL
rotation2.txt: 22 Rotationen, 125 Zustände gesehen
RRRRLRRLLLLLLLRRRRRRLR
rotation3.txt: 90 Rotationen, 180367 Zustände gesehen
LRRRRLLLRRRLLLLLLLRRLLRRRRRRRLRRRRRRRRRRRLLLRRRRRLLLLLRRRRLLRRRRRRRLRLLLLLLLLLLLRLLRRRRRLR


Das sind genau die Werte der Lösungshinweise: **6**, **22** und **90 Rotationen**. Auch die Rotationsfolgen aus den Hinweisen stimmen Zeichen für Zeichen mit unseren überein. Es kann mehrere gleich kurze Lösungen geben; wir probieren in jedem Zustand `L` vor `R`, und das tun offenbar auch die Hinweise.

Bei `rotation3.txt` zeigt sich der Sinn der Merkliste: Statt $2^{90} > 10^{27}$ Folgen gibt es nur gut 180 000 verschiedene Zustände.

**Kontrolle der Lösung:** Wir spielen die Folgen für die drei Beispiele noch einmal von Hand durch, mit einer Funktion, die nicht die Breitensuche benutzt: Nach jeder Rotation darf noch **kein** Stäbchen draußen sein, nach der letzten muss eines draußen sein.

In [ ]:
def pruefe_folge(start, folge):
    """Spielt die Drehfolge nach. Wahr, wenn erst nach der letzten Rotation ein Stäbchen draußen ist."""
    zustand = start
    for richtung in folge:
        if ist_fertig(zustand):
            return False
        zustand = drehung(zustand, richtung)
    return ist_fertig(zustand)


for nummer in (1, 2, 3):
    start = lies_puzzle('beispieldaten/rotation' + str(nummer) + '.txt')
    print('rotation' + str(nummer) + ':', pruefe_folge(start, ergebnisse[nummer][0]))

rotation1: True
rotation2: True
rotation3: True


## Ein Puzzle ohne Lösung und eine Rotation

Das Puzzle `ohne_loesung.txt` ist das Puzzle ohne Lösung aus den Lösungshinweisen: Alle Stäbchen sind waagerecht, die Lücke liegt unten. Dreht man den Rahmen, so liegen die Stäbchen senkrecht, aber dann ist die Lücke seitlich oder oben. **Ein Stäbchen, das parallel zur Wand der Lücke liegt, kann nie durch die Lücke.** Die Breitensuche sieht das daran, dass nach nur 17 verschiedenen Zuständen die Warteschlange leer ist. `eine_rotation.txt` ist ein Puzzle, bei dem eine einzige Rotation nach rechts genügt.

In [ ]:
ohne = lies_puzzle('eigene_testdaten/ohne_loesung.txt')
zeige(ohne)
print()
zeige_loesung(ohne, False)
print()
eine = lies_puzzle('eigene_testdaten/eine_rotation.txt')
zeige_loesung(eine, True)

##########
#        #
#        #
#        #
#        #
#  555555#
#444     #
#  6666  #
#111 2222#
#### #####

Es gibt keine Lösung! Es wurden alle 17 Zustände ausprobiert.

Es wurde eine Lösung mit 1 Rotationen gefunden: R
######
#    #
#    #
#    #
#00   
######

Rotation nach rechts:
######
#    #
#    #
#    #
#0   #
#0####


**Zusammenfassung:** Wir haben in jedem Zustand die beiden Drehungen ausprobiert (jeweils mit Schwerkraft), jeden neuen Zustand nur einmal betrachtet und in der Reihenfolge der Entfernung vom Start abgearbeitet. Deshalb ist die erste gefundene Lösung kürzest möglich, und ohne Lösung endet das Programm, sobald alle erreichbaren Zustände gesehen sind.

## Für Neugierige: Wie viele Zustände gibt es wirklich?

Die Breitensuche kann bei großen Puzzles trotz Merkliste an die Grenzen des Arbeitsspeichers stoßen: Bei `rotation3.txt` sind es gut 180 000 Zustände mit je 100 Zeichen, das ist noch klein. Für größere Rahmen mit mehr Stäbchen werden es schnell Millionen. Zwei Ideen, wie man das verkleinern kann:

- **Zustand kompakter speichern:** Statt des ganzen Bildes (n × n Zeichen) genügt die Lage jedes Stäbchens (Zeile, Spalte, Ausrichtung). Der Rahmen kann dann als Zahl oder kurzes Tupel im `dict` stehen.
- **Weniger Drehungen prüfen:** Dreht man links und danach sofort rechts, kommt man (bis auf das Fallen) zurück; die Merkliste fängt das schon ab, aber man könnte solche Paare gleich weglassen.

Für die Aufgabe genügt aber die einfache Version, wie die Lösungshinweise sagen: „Das Programm terminiert sehr schnell.“

## Zum Schluss: Dasselbe mit MiniZinc?

Die Aufgabe ist ein **kürzester Weg in einem Zustandsgraphen**, und das ist für die Breitensuche ideal. Mit MiniZinc muss man dagegen die Länge `T` der Drehfolge **festlegen** und für jede Rotation `t` die Lage aller Stäbchen als Variablen anlegen (der „Film“ der Drehungen steht im Modell). Wir haben es trotzdem gemessen, damit du die Grenze siehst.

**Das Modell:** Jedes Stäbchen hat nach jeder Rotation eine oberste Zeile `r`, eine linke Spalte `c` und eine Ausrichtung `hz` (waagerecht oder nicht); die Lücke wandert als `(gr, gc)` mit. Die Variable `zug[t]` wählt links oder rechts.

- **Drehen** sind Formeln: Ein Feld `(z, s)` wandert bei einer Linksrotation nach `(n-1-s, z)`, bei einer Rechtsrotation nach `(s, n-1-z)`. Die Variable `hz` kippt bei jeder Rotation.
- **Fallen** wird nicht simuliert, sondern durch **Bedingungen** beschrieben: Ein Stäbchen rutscht nur nach unten (`r >= rr`), zwei Stäbchen in derselben Spalte behalten ihre Reihenfolge (sie können einander nicht überholen), und am Ende liegt jedes Stäbchen auf dem Boden oder direkt auf einem anderen. Das legt den Endzustand eindeutig fest.
- **Ziel:** Nach der letzten Rotation steht ein senkrechtes Stäbchen in der Lücke unten.
- Die Bedingungen benutzen `forall`, `exists`, **Variablen als Index** (wie in den Arbeitsblättern zu MiniZinc), einen kleinen `predicate` und `let` mit Hilfsvariablen. Als Daten stehen Größe, Stäbchenlängen, Startlagen und Lücke in der `.dzn`-Datei.
- Das **kürzeste** `T` findet man, indem man `T = 1, 2, 3, …` nacheinander ausprobiert: Das erste `T`, bei dem das Modell erfüllbar ist, ist die Antwort.

```minizinc
int: n;                                      % Größe des Rahmens
int: k;                                      % Zahl der Stäbchen
int: T;                                      % Zahl der Rotationen (wird durchprobiert)
array[1..k] of int: len;                     % Länge jedes Stäbchens
array[1..k] of int: r0;                      % Startlage: oberste Zeile ...
array[1..k] of int: c0;                      % ... linke Spalte ...
array[1..k] of bool: h0;                     % ... waagerecht?
int: g0r;                                    % Lücke im Rand (Zeile, Spalte)
int: g0c;

array[1..T] of var 1..2: zug;                % 1 = links, 2 = rechts
array[0..T, 1..k] of var 0..n-1: r;          % Lage nach dem Fallen
array[0..T, 1..k] of var 0..n-1: c;
array[0..T, 1..k] of var bool: hz;
array[0..T] of var 0..n-1: gr;               % Lage der Lücke nach der Rotation
array[0..T] of var 0..n-1: gc;
array[1..T, 1..k] of var 0..n-1: rr;         % Lage nach dem Drehen, vor dem Fallen
array[1..T, 1..k] of var 0..n-1: rc;

constraint gr[0] = g0r /\ gc[0] = g0c;
constraint forall(i in 1..k)(r[0,i] = r0[i] /\ c[0,i] = c0[i] /\ hz[0,i] = h0[i]);

% Drehen: Felder (z, s) wandern bei links nach (n-1-s, z), bei rechts nach (s, n-1-z)
constraint forall(t in 1..T)(
  (zug[t] = 1 -> gr[t] = n-1-gc[t-1] /\ gc[t] = gr[t-1]) /\
  (zug[t] = 2 -> gr[t] = gc[t-1] /\ gc[t] = n-1-gr[t-1]));
constraint forall(t in 1..T, i in 1..k)(
  hz[t,i] = not hz[t-1,i] /\
  (zug[t] = 1 -> (if hz[t-1,i] then rr[t,i] = n-c[t-1,i]-len[i] /\ rc[t,i] = r[t-1,i]
                  else rr[t,i] = n-1-c[t-1,i] /\ rc[t,i] = r[t-1,i] endif)) /\
  (zug[t] = 2 -> (if hz[t-1,i] then rr[t,i] = c[t-1,i] /\ rc[t,i] = n-1-r[t-1,i]
                  else rr[t,i] = c[t-1,i] /\ rc[t,i] = n-r[t-1,i]-len[i] endif)));

% Fallen: nur nach unten, die Reihenfolge übereinander bleibt, am Ende liegt jedes Stäbchen auf
predicate ueberlappt(int: t, int: i, int: j) =        % teilen sich i und j eine Spalte?
  c[t,j] <= c[t,i] + (if hz[t,i] then len[i] else 1 endif) - 1 /\
  c[t,i] <= c[t,j] + (if hz[t,j] then len[j] else 1 endif) - 1;

constraint forall(t in 1..T, i in 1..k)(
  c[t,i] = rc[t,i] /\ r[t,i] >= rr[t,i] /\
  (let { var int: ht = if hz[t,i] then 1 else len[i] endif;
         % unterste erlaubte Zeile: n-2, in der Lücke (senkrecht darüber) n-1
         var int: lim = if gr[t] = n-1 /\ (not hz[t,i] \/ len[i] = 1) /\ c[t,i] = gc[t]
                        then n-1 else n-2 endif } in
   r[t,i] + ht - 1 <= lim /\
   (r[t,i] + ht - 1 = lim \/                                  % liegt auf dem Boden ...
    exists(j in 1..k where j != i)(                           % ... oder auf einem anderen Stäbchen
      ueberlappt(t, i, j) /\ rr[t,i] < rr[t,j] /\ r[t,i] + ht = r[t,j]))));
constraint forall(t in 1..T, i in 1..k, j in 1..k where i != j)(
  (ueberlappt(t, i, j) /\ rr[t,i] < rr[t,j])
  -> r[t,i] + (if hz[t,i] then 1 else len[i] endif) <= r[t,j]);

% Ziel: nach der letzten Rotation steht ein Stäbchen in der Lücke
constraint exists(i in 1..k)(gr[T] = n-1 /\ (not hz[T,i] \/ len[i] = 1) /\ c[T,i] = gc[T]
                             /\ r[T,i] + len[i] - 1 = n-1);
solve satisfy;
output [ if fix(zug[t]) = 1 then "L" else "R" endif | t in 1..T ] ++ ["\n"];
```

Eine Python-Zelle erzeugt die Datendatei aus dem eingelesenen Puzzle:

In [ ]:
def als_dzn(puzzle, T):
    """MiniZinc-Daten (.dzn) für ein Puzzle und die Zahl T der Rotationen."""
    n = len(puzzle)
    felder = {}
    for r in range(n):
        for c in range(n):
            if puzzle[r][c] not in ' #':
                felder.setdefault(puzzle[r][c], []).append((r, c))
    ziffern = sorted(felder)
    zeilen = [min(r for r, c in felder[z]) for z in ziffern]
    spalten = [min(c for r, c in felder[z]) for z in ziffern]
    laengen = [len(felder[z]) for z in ziffern]
    # waagerecht, wenn alle Felder in derselben Zeile liegen (Länge 1 zählt auch als waagerecht)
    waagerecht = ['true' if len({r for r, c in felder[z]}) == 1 else 'false' for z in ziffern]
    # Die Lücke: das Leerzeichen im Rand
    luecke = [(r, c) for r in range(n) for c in range(n)
              if (r in (0, n - 1) or c in (0, n - 1)) and puzzle[r][c] == ' '][0]
    return ('n = ' + str(n) + ';\nk = ' + str(len(ziffern)) + ';\nT = ' + str(T) + ';\n'
            + 'len = ' + str(laengen) + ';\nr0 = ' + str(zeilen) + ';\nc0 = ' + str(spalten) + ';\n'
            + 'h0 = [' + ', '.join(waagerecht) + '];\n'
            + 'g0r = ' + str(luecke[0]) + ';\ng0c = ' + str(luecke[1]) + ';')


print(als_dzn(start1, 6))

n = 8;
k = 7;
T = 6;
len = [2, 2, 4, 2, 3, 2, 3];
r0 = [1, 3, 3, 4, 4, 5, 6];
c0 = [6, 1, 3, 1, 6, 1, 1];
h0 = [false, true, true, true, false, true, true];
g0r = 7;
g0c = 3;


Ausgabe für `rotation1.txt` mit Gecode, `T` von 1 bis 6 (jede Zeile ein eigener Lauf, alle zusammen in unter einer Sekunde):

```
T = 1: keine Lösung
T = 2: keine Lösung
T = 3: keine Lösung
T = 4: keine Lösung
T = 5: keine Lösung
T = 6: LLLLRL
```

Bei `T = 6` findet das Modell die Folge `LLLLRL`, und für `T = 1` bis `5` beweist es, dass es nicht kürzer geht. Das stimmt mit unserer Breitensuche überein.

**Die Grenze:** Schon beim nächsten Beispiel wird es zäh. Bei `rotation2.txt` (kürzeste Lösung 22) braucht der Solver Chuffed für den Beweis „mit 11 bis 17 Rotationen geht es nicht“ je 1,5 bis 15 Sekunden, bei `T = 18` ist er nach 20 Sekunden nicht fertig, und auch mit `T = 22` findet er die Lösung in 20 Sekunden nicht. Bei `rotation3.txt` (`T = 90`) kommt er in keiner sinnvollen Zeit zu einem Ergebnis, schon das Aufstellen des Modells mit 90 Rotationen × 10 Stäbchen dauert lange. Die Breitensuche löst alle drei Beispiele in Sekunden: Sie sieht jeden Zustand nur **einmal**, während ein Constraint-Solver die Zustände nach jeder Rotation getrennt als Variablen behandelt und die Wiederholungen nicht ausnutzt. Für solche Kürzeste-Weg-Aufgaben in einem **endlichen Zustandsgraphen** bleibt die Breitensuche die bessere Wahl; MiniZinc eignet sich eher, wenn man Bedingungen auf einmal beschreiben kann, ohne die Zeit zu entrollen.